In [7]:
"""
DeepCAVE API mode minimal end-to-end test.

Flow:
    ConfigSpace (search space)
        -> Recorder (record trials, DeepCAVE native format)
        -> DeepCAVERun.from_path (load run back)
        -> CostOverTime plugin (draw a Plotly figure, no surrogate model)

The objective below is a cheap synthetic function on purpose, so this test
finishes in seconds. Replace ``evaluate()`` with your real training/evaluation
code when you build the actual hyperparameter search scheme.

Run from the ``deepcave`` conda environment:

    conda activate deepcave
    python deepcave_api_test.py
"""

'\nDeepCAVE API mode minimal end-to-end test.\n\nFlow:\n    ConfigSpace (search space)\n        -> Recorder (record trials, DeepCAVE native format)\n        -> DeepCAVERun.from_path (load run back)\n        -> CostOverTime plugin (draw a Plotly figure, no surrogate model)\n\nThe objective below is a cheap synthetic function on purpose, so this test\nfinishes in seconds. Replace ``evaluate()`` with your real training/evaluation\ncode when you build the actual hyperparameter search scheme.\n\nRun from the ``deepcave`` conda environment:\n\n    conda activate deepcave\n    python deepcave_api_test.py\n'

用于测试环境（API实现路径）

In [4]:
from pathlib import Path

import numpy as np
import ConfigSpace as CS

from deepcave import Objective, Recorder
from deepcave.runs.converters.deepcave import DeepCAVERun
from deepcave.plugins.objective.cost_over_time import CostOverTime


BASE_DIR = '/Users/crystal/Desktop/研一上/ML/deepcave-api-test'
LOG_DIR = BASE_DIR + "/logs"
OUTPUT_PNG = BASE_DIR + "/cost_over_time.png"

N_CONFIGS = 30
SEED = 42
RNG = np.random.default_rng(SEED)

In [5]:

def build_configspace() -> CS.ConfigurationSpace:
    """A small 3-parameter search space used only for testing."""
    configspace = CS.ConfigurationSpace(seed=SEED)
    configspace.add(
        [
            CS.hyperparameters.UniformFloatHyperparameter(
                name="alpha", lower=0.01, upper=1.0, log=True
            ),
            CS.hyperparameters.UniformFloatHyperparameter(name="beta", lower=0.0, upper=1.0),
            CS.hyperparameters.CategoricalHyperparameter(
                name="gamma", choices=[0.0, 1.0]
            ),
        ]
    )
    return configspace


def evaluate(config: CS.Configuration) -> float:
    """
    Cheap stand-in for a real objective.

    Realistic-looking cost in [0, 1]: smaller alpha/gamma and beta close to 0.5
    are better, plus a little noise so the trajectory plot is not a flat line.
    """
    alpha = config["alpha"]
    beta = config["beta"]
    gamma = config["gamma"]

    cost = 0.4 * alpha + 0.8 * abs(beta - 0.5) + 0.2 * abs(gamma - 1.0)
    cost += float(RNG.normal(0.0, 0.01))
    return float(np.clip(cost, 0.0, 1.0))



In [ ]:
# 1. Define the search space
configspace = build_configspace()

# 2. Define a single minimization objective in [0, 1]
objective = Objective("score", lower=0, upper=1, optimize="lower")

# 3. Record a fake "hyperparameter search" run in DeepCAVE native format
# LOG_DIR.mkdir(exist_ok=True)
with Recorder(
    configspace,
    objectives=[objective],
    save_path=str(LOG_DIR),
    prefix="run",
    overwrite=True,
) as recorder:
    for config in configspace.sample_configuration(N_CONFIGS):
        recorder.start(config, budget=1, seed=SEED)
        recorder.end(costs=[evaluate(config)], seed=SEED)
    run_path = recorder.path

print(f"Recorded run at: {run_path}")

# 4. Load the run back through the native converter
run = DeepCAVERun.from_path(Path(run_path))
print(f"Run name: {run.name}")
print(f"Trial count: {len(run.history)}")
print(f"Objectives: {run.get_objective_ids()}")
print(f"Budgets: {run.get_budget_ids()}")

# 5. Use the lightest plugin: Cost Over Time (no surrogate model training)
plugin = CostOverTime()
inputs = plugin.generate_inputs(
    objective_id=run.get_objective_ids()[0],
    budget_id=run.get_budget_ids()[-1],
    xaxis="trials",  # plot vs number of evaluated trials instead of wall time
    show_runs=True,
    show_groups=True,
)
outputs = plugin.generate_outputs(run, inputs)
figure = plugin.load_outputs([run], inputs, outputs)

figure.write_image(str(OUTPUT_PNG), width=1000, height=500, scale=2)
print(f"Figure saved to: {OUTPUT_PNG}")


Recorded run at: /Users/crystal/Desktop/研一上/ML/deepcave-api-test/logs/run
Run name: run
Trial count: 30
Objectives: [0]
Budgets: [0]
Figure saved to: /Users/crystal/Desktop/研一上/ML/deepcave-api-test/cost_over_time.png
